# 1. 배치 ID 분류와 공유 후보 scorer를 실제로 비교합니다

[고정 명세](../../../ai-harness/ml-experiments/experiments/human-placement-candidate-scoring.md)의 모델 비교 단계입니다. Coverage가 Train6,417/Validation1,516구간 모두100%로 확인됐으므로 같은 구간을 유지합니다.

- **ID 원래 출력:** 기존207→64→2,133-class 모델을 실제 학습합니다.
- **ID 후보 제한:** 위와 같은 weight로 후보와 Train 어휘의 교집합만 순위화합니다. 추가 모델은 없습니다.
- **공유 후보 scorer:**407→64→이진 MLP를 실제로 학습하고 모든 후보를 점수화합니다.

별도 Python 실행은 필요 없습니다. 두 모델 모두 실제 학습 코드가 있으며, Full 스위치만 기본 False입니다. 기존 coverage Notebook의 실행 출력·설정은 수정하지 않습니다. 저장된 evidence를 읽으므로 **Node/후보 생성은 다시 호출하지 않습니다.**


## 2. 선택지와 정답은 다릅니다

후보가 P/Q/R이고 Human 선택이 Q라면 scorer label은0/1/0입니다. 학습 weight는1/4,1/2,1/4로 합1입니다. 후보가 더 많아져도 한 구간의 영향력 합은1입니다. Scaler weight는 정답과 관계없이 각1/3입니다.

ID 방식에서 Q가 미관측 key면 정답을 선택할 수 없습니다. Scorer는 Q에도 점수를 줄 수 있지만, 잘 고르는지는 Full 결과로만 판단합니다. 정답 목표/휴리스틱 score/최종 pose를 입력에 주지 않습니다.


In [ ]:
import copy
import gc
import hashlib
import inspect
import json
import math
import os
import platform
import tempfile
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-candidate-scoring"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_candidate_model_comparison.ipynb"
SNAPSHOT_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_snapshot_baseline.ipynb"
SNAPSHOT_SHA = "233ceca8322bd4335b798cdd452064d389e0ddbce6451610c7ba7d56a1dd9c5c"
COVERAGE_SHA = "c6575ebaa63744cd2891096bb9ad37cf3d6206b488bf65ca038d8df29e861d8d"
EVIDENCE_SHA = "fcad6113509eab114f3a1639ef59b6f26071df22edf8d17273fa21c43026b684"
REFERENCE_SHA = "d87456bbb3d4f289db38d1bc70d8b6b8d1615f34a5bd4ddaa3f025d40f6fb19e"
COVERAGE_DEFS_SHA = "d3846c88f1f6656536e5f0247ac87e0ae61156145d1baf61aefa53e2786a1e18"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(SNAPSHOT_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == SNAPSHOT_SHA
shared = {"__name__": "candidate_models_shared"}
for cell in selected:
    exec(compile(cell["source"], str(SNAPSHOT_NOTEBOOK), "exec"), shared)
shared["EXPERIMENT_ID"], shared["SOURCE_NOTEBOOK"] = EXPERIMENT_ID, SOURCE_NOTEBOOK
shared["base"]["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "repo_input",
             "load_snapshots", "physical_key", "report_path", "publish_report", "reject_constant",
             "finite_float", "threadpool_limits", "predict_probabilities"):
    globals()[name] = shared[name]
STRATEGIES = ("id_unmasked", "id_candidate_masked", "shared_candidate_score")
FULL_EPOCHS = 30
EXPECTED_ROWS = {"train": 6417, "validation": 1516}
EXPECTED_CANDIDATES = {"train": 147450, "validation": 34552}
require("sample_weight" in inspect.signature(MLPClassifier.partial_fit).parameters
        and "sample_weight" in inspect.signature(StandardScaler.fit).parameters,
        "WEIGHTED_ESTIMATOR_API_REQUIRED")


In [ ]:
def read_model_inputs():
    """Read only pinned offline Train/Validation and accepted saved candidates; no Node or Test."""
    directory = "data/ml-experiments/human-placement-candidate-scoring/"
    paths = {name: repo_input(os.environ.get(env, default)) for name, env, default in (
        ("coverage", "HUMAN_CANDIDATE_COVERAGE_INPUT", directory + "coverage-report-run1.json"),
        ("evidence", "HUMAN_CANDIDATE_EVIDENCE_INPUT", directory + "candidate-evidence-run1.json"),
        ("reference", "HUMAN_CANDIDATE_REFERENCE",
         "data/ml-experiments/human-placement-snapshot-baseline/full-report-run2.json"))}
    watched = {p: sha256_file(p) for p in paths.values()}
    require([watched[paths[k]] for k in ("coverage", "evidence", "reference")]
            == [COVERAGE_SHA, EVIDENCE_SHA, REFERENCE_SHA], "PINNED_REPORT_OR_EVIDENCE_CHANGED")
    coverage, evidence, reference = [json.loads(paths[k].read_text(encoding="utf-8"),
        parse_constant=reject_constant, parse_float=finite_float) for k in ("coverage", "evidence", "reference")]
    require(coverage["experimentId"] == EXPERIMENT_ID and coverage["runMode"] == "full"
            and coverage["analysisStatus"] == "candidate_coverage_requires_review"
            and coverage["candidateContract"] == "snapshot_static_supported_cells_v1"
            and coverage["fullCohortCovered"] is True and coverage["modelFitCount"] == 0
            and coverage["scalerFitCount"] == 0 and coverage["gameIdLeakage"] == 0
            and coverage["evidence"]["sha256"] == EVIDENCE_SHA
            and coverage["provenance"]["sourceDefinitionSha256"] == COVERAGE_DEFS_SHA,
            "COVERAGE_CONTRACT_INVALID")
    for relative, digest in coverage["provenance"]["coreFiles"].items():
        path = repo_input(relative)
        require(sha256_file(path) == digest, "CACHED_CORE_IDENTITY_CHANGED")
        watched[path] = digest
    require(reference["experimentId"] == "human-placement-snapshot-baseline"
            and reference["runMode"] == "full" and reference["fullCohortCovered"] is True
            and reference["sourceDefinitionSha256"] == SNAPSHOT_SHA, "REFERENCE_CONTRACT_INVALID")
    data, sources = load_snapshots()
    watched.update(sources)
    require(coverage["provenance"]["manifestSha256"] == shared["MANIFEST_SHA"]
            and coverage["provenance"]["datasetSha256"]
                == {s: shared["EXPECTED"][s][-1] for s in EXPECTED_ROWS}, "COVERAGE_DATASET_MISMATCH")
    require(len(evidence) == len(coverage["segments"]) == sum(EXPECTED_ROWS.values()),
            "EVIDENCE_COHORT_INVALID")
    vocabulary = shared["train_vocabulary"](data["train"])
    require(len(vocabulary) == 2133
            and hashlib.sha256(json_bytes(vocabulary)).hexdigest()
                == "26edffc2ac5501450f78e976818cba195882d558c41530ff2196a0b017d45084",
            "BASELINE_VOCABULARY_CHANGED")
    grouped = {s: [] for s in EXPECTED_ROWS}
    ordered_rows = data["train"] + data["validation"]
    for item, diagnostic, row in zip(evidence, coverage["segments"], ordered_rows):
        require(item["segmentId"] == diagnostic["segmentId"] == row["segmentId"]
                and item["split"] == diagnostic["split"] == row["split"]
                and hashlib.sha256(json_bytes(item["candidates"])).hexdigest()
                    == diagnostic["candidateSetSha256"], "EVIDENCE_JOIN_OR_HASH_MISMATCH")
        candidates = item["candidates"]
        require(len(candidates) == diagnostic["candidates"] >= 2
                and diagnostic["choiceEligible"] is True and diagnostic["targetIncluded"] is True,
                "APPROVED_COMMON_COHORT_CHANGED")
        keys = []
        for c in candidates:
            key = physical_key({"pieceType": row["pieceType"], **c["representative"], "cells": c["cells"]},
                               selected=True)
            require(key == c["key"] and type(c["aliasCount"]) is int
                    and 1 <= c["aliasCount"] <= 4, "CANDIDATE_KEY_OR_ALIAS_INVALID")
            keys.append(key)
        require(keys == sorted(set(keys)) and row["targetKey"] in keys
                and diagnostic["baselineTargetOov"] == (row["targetKey"] not in vocabulary),
                "CANDIDATE_TARGET_OR_ORDER_INVALID")
        grouped[row["split"]].append(candidates)
    for split in EXPECTED_ROWS:
        require(len(grouped[split]) == EXPECTED_ROWS[split]
                and sum(map(len, grouped[split])) == EXPECTED_CANDIDATES[split],
                "CANDIDATE_COUNTS_CHANGED")
    require(all(sha256_file(p) == digest for p, digest in watched.items()), "INPUT_CHANGED")
    return data, grouped, vocabulary, reference, coverage, watched


In [ ]:
def pack_candidates(rows, candidate_lists):
    """Use only board/piece and candidate cells for X; Human target supplies y/weights separately."""
    require(rows and len(rows) == len(candidate_lists), "PACK_COHORT_INVALID")
    counts = np.array([len(c) for c in candidate_lists], dtype=int)
    require((counts >= 2).all(), "CHOICE_REQUIRED_COHORT_EXPECTED")
    offsets = np.concatenate(([0], np.cumsum(counts)))
    n = int(offsets[-1])
    x = np.zeros((n, 407), dtype=np.float64)
    y = np.zeros(n, dtype=int)
    weight, scaler_weight = np.empty(n), np.empty(n)
    keys = []
    for i, (row, candidates) in enumerate(zip(rows, candidate_lists)):
        a, b = int(offsets[i]), int(offsets[i + 1])
        require(row["x"].shape == (207,) and np.isfinite(row["x"]).all()
                and np.isin(row["x"], [0, 1]).all(), "RAW_INPUT_INVALID")
        x[a:b, :207] = row["x"]
        row_keys = [c["key"] for c in candidates]
        require(row_keys == sorted(set(row_keys)) and row["targetKey"] in row_keys, "CANDIDATE_LABEL_INVALID")
        keys.append(row_keys)
        for j, candidate in enumerate(candidates):
            cells = candidate["cells"]
            key = physical_key({"pieceType": row["pieceType"], **candidate["representative"], "cells": cells},
                               selected=True)
            require(key == candidate["key"], "CANDIDATE_FORMAT_INVALID")
            x[a + j, [207 + cell["row"] * 10 + cell["col"] for cell in cells]] = 1
        target = a + row_keys.index(row["targetKey"])
        y[target] = 1
        weight[a:b] = 0.5 / (len(candidates) - 1)
        weight[target] = 0.5
        scaler_weight[a:b] = 1 / len(candidates)
        require(np.isclose(weight[a:b].sum(), 1, atol=1e-12)
                and np.isclose(scaler_weight[a:b].sum(), 1, atol=1e-12), "DECISION_WEIGHT_INVALID")
    require(np.isfinite(x).all() and np.isfinite(weight).all() and (weight > 0).all()
            and y.sum() == len(rows) and np.isclose(weight.sum(), len(rows), atol=1e-9),
            "PACK_VALUES_INVALID")
    return {"x": x, "y": y, "weight": weight, "scalerWeight": scaler_weight,
            "offsets": offsets, "keys": keys, "rows": rows}

def scale_candidate_inputs(train, validation):
    """Fit the weighted scaler on Train only; transform the large matrices in place."""
    require(all(r["split"] == "train" for r in train["rows"])
            and all(r["split"] == "validation" for r in validation["rows"]), "SCALER_SPLIT_INVALID")
    scaler = StandardScaler().fit(train["x"], sample_weight=train["scalerWeight"])
    require(np.isfinite(scaler.mean_).all() and np.isfinite(scaler.scale_).all()
            and (scaler.scale_ > 0).all(), "SCALER_INVALID")
    train["x"] = scaler.transform(train["x"], copy=False)
    validation["x"] = scaler.transform(validation["x"], copy=False)
    return scaler

# Repeated 1/K sums accumulate floating-point error; model-metric tolerance stays 1e-12.
SCALER_COMPARISON_ATOL = 1e-10

def check_board_piece_scalers(id_scaler, candidate_scaler):
    mean_error = float(np.max(np.abs(candidate_scaler.mean_[:207] - id_scaler.mean_)))
    scale_error = float(np.max(np.abs(candidate_scaler.scale_[:207] - id_scaler.scale_)))
    require(np.isfinite([mean_error, scale_error]).all()
            and max(mean_error, scale_error) <= SCALER_COMPARISON_ATOL,
            f"BOARD_PIECE_SCALER_WEIGHTING_CHANGED: mean={mean_error:.3e}, scale={scale_error:.3e}")


In [ ]:
def scored_record(row, keys, scores):
    require(len(keys) == len(scores) and len(set(keys)) == len(keys)
            and np.isfinite(scores).all(), "RANK_SCORE_INVALID")
    order = sorted(range(len(keys)), key=lambda i: (-float(scores[i]), keys[i]))
    ordered = [keys[i] for i in order]
    rank = ordered.index(row["targetKey"]) + 1 if row["targetKey"] in ordered else None
    return {"segmentId": row["segmentId"], "gameId": row["gameId"], "pieceType": row["pieceType"],
            "rank": rank, "knownTarget": rank is not None, "knownTargetNll": None,
            "logClipped": False, "predictedKey": ordered[0] if ordered else None,
            "selectionAvailable": bool(ordered)}

def topk_result(scored):
    """Reuse house ranking arithmetic; discard vocabulary/NLL fields with different meanings."""
    result = shared["evaluate_rankings"](scored)
    columns = ("segments", "top1", "top3", "mrr", "selectionUnavailableSegments")
    return {"gameMacro": {k: result["gameMacro"][k] for k in ("top1", "top3", "mrr")},
            "segmentMicro": {k: result["segmentMicro"][k] for k in columns},
            "perGame": [{"gameId": r["gameId"], **{k: r[k] for k in columns}} for r in result["perGame"]],
            "perPiece": [{"pieceType": r["pieceType"], **{k: r[k] for k in columns}} for r in result["perPiece"]]}

def masked_id_rankings(rows, probability, vocabulary, candidate_lists):
    require(probability.shape == (len(rows), len(vocabulary)), "ID_PROBABILITY_SHAPE")
    lookup = {key: i for i, key in enumerate(vocabulary)}
    result = []
    for row, p, candidates in zip(rows, probability, candidate_lists):
        keys = [c["key"] for c in candidates if c["key"] in lookup]
        result.append(scored_record(row, keys, np.array([p[lookup[k]] for k in keys])))
    return result

def binary_rankings(pack, probability):
    require(probability.shape == (len(pack["y"]),) and np.isfinite(probability).all()
            and ((0 <= probability) & (probability <= 1)).all(), "BINARY_PROBABILITY_INVALID")
    return [scored_record(row, pack["keys"][i],
                         probability[pack["offsets"][i]:pack["offsets"][i + 1]])
            for i, row in enumerate(pack["rows"])]

def binary_nll(pack, probability):
    p = np.clip(probability, 1e-15, 1 - 1e-15)
    require(probability.shape == pack["y"].shape and np.isfinite(probability).all()
            and ((0 <= probability) & (probability <= 1)).all(), "BINARY_LOSS_INPUT_INVALID")
    loss = -(pack["y"] * np.log(p) + (1 - pack["y"]) * np.log1p(-p))
    return float(np.average(loss, weights=pack["weight"]))

def predict_positive(model, x):
    require(np.array_equal(model.classes_, np.array([0, 1])), "BINARY_CLASS_ORDER_INVALID")
    values = np.empty(len(x))
    for start in range(0, len(x), 8192):
        p = model.predict_proba(x[start:start + 8192])
        require(p.shape == (min(8192, len(x) - start), 2) and np.isfinite(p).all()
                and (p >= 0).all() and np.allclose(p.sum(axis=1), 1, atol=1e-12), "MODEL_PROBABILITY_INVALID")
        values[start:start + len(p)] = p[:, 1]
    return values

def verify_baseline(result, reference, loss, history):
    original = reference["results"]["board_piece"]
    error = max(abs(result[group][metric] - original[group][metric])
                for group, metrics in (("gameMacro", ("top1", "top3", "mrr")),
                                       ("segmentMicro", ("top1", "top3", "mrr", "knownTargetNll")))
                for metric in metrics)
    require(error <= 1e-12 and np.allclose(loss, reference["trainingLoss"]["board_piece"], atol=1e-12, rtol=0)
            and len(history) == 30, "ID_BASELINE_NOT_REPRODUCED_SHARED_FIT_NOT_STARTED")
    for a, b in zip(history, reference["epochMetrics"]["board_piece"]):
        require(a["epoch"] == b["epoch"] and all(np.isclose(a[split][k], b[split][k], atol=1e-12, rtol=0)
                for split in ("train", "validation") for k in ("top1", "top3", "mrr", "knownTargetNll")),
                "ID_HISTORY_NOT_REPRODUCED")
    return error


In [ ]:
def fit_shared_scorer(train, validation, epochs):
    """Real weighted partial_fit; validation is monitoring only, with no checkpoint selection."""
    require(all(r["split"] == "train" for r in train["rows"])
            and all(r["split"] == "validation" for r in validation["rows"])
            and type(epochs) is int and epochs > 0, "FIT_SCOPE_INVALID")
    model = MLPClassifier(hidden_layer_sizes=(64,), activation="relu", solver="adam",
                          alpha=0.0001, batch_size=min(128, len(train["y"])), learning_rate_init=0.001,
                          shuffle=True, random_state=1729, early_stopping=False)
    history, optimization_loss = [], []
    for epoch in range(epochs):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.partial_fit(train["x"], train["y"], sample_weight=train["weight"],
                              classes=np.array([0, 1]) if epoch == 0 else None)
        require(not caught, "UNEXPECTED_FIT_WARNING: " + "; ".join(
            f"{w.category.__name__}: {w.message}" for w in caught))
        require(np.isfinite(model.loss_) and model.t_ == (epoch + 1) * len(train["y"]),
                "NONFINITE_LOSS_OR_UPDATE_BUDGET_CHANGED")
        optimization_loss.append(float(model.loss_))
        current = {"epoch": epoch + 1}
        for split, pack in (("train", train), ("validation", validation)):
            probability = predict_positive(model, pack["x"])
            current[split] = {**topk_result(binary_rankings(pack, probability))["segmentMicro"],
                              "weightedBinaryNll": binary_nll(pack, probability)}
        history.append(current)
        print(f"shared scorer epoch {epoch + 1}/{epochs}: Train BCE={current['train']['weightedBinaryNll']:.6f}, "
              f"Val BCE={current['validation']['weightedBinaryNll']:.6f}, "
              f"Val placement Top-1={current['validation']['top1']:.4f}")
    require(all(np.isfinite(x).all() for x in (*model.coefs_, *model.intercepts_)), "NONFINITE_MODEL")
    return model, optimization_loss, history

def run_model_comparison(output_name):
    """Publish one completed report after two actual fits, baseline reproduction and input rechecks."""
    output = report_path(output_name)
    require((platform.python_version(), np.__version__, sklearn.__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_TRAINING_ENVIRONMENT_REQUIRED")
    source_sha = definition_sha(SOURCE_NOTEBOOK)
    data, candidates, vocabulary, reference, coverage, watched = read_model_inputs()
    print("Full comparison: actual ID model + actual shared scorer; 30 epochs each. Masked ID reuses ID weights.")
    print("Train/Validation candidates:", EXPECTED_CANDIDATES, "; no exclusions, no Node invocation.")
    key_to_id = {key: i for i, key in enumerate(vocabulary)}
    x = np.stack([r["x"] for r in data["train"]])
    validation_x = np.stack([r["x"] for r in data["validation"]])
    y = np.array([key_to_id[r["targetKey"]] for r in data["train"]])
    id_scaler = StandardScaler().fit(x)
    train_id = id_scaler.transform(x)
    validation_id = id_scaler.transform(validation_x)
    with threadpool_limits(limits=1):
        started = time.perf_counter()
        id_model, id_loss, id_history, id_ranked = shared["fit_with_validation"](
            train_id, y, validation_id, data["train"], data["validation"], vocabulary, FULL_EPOCHS)
        print(f"ID fit/epoch-monitoring elapsed: {time.perf_counter() - started:.2f}s (non-canonical timing)")
        require(id_model.t_ == len(y) * FULL_EPOCHS, "ID_UPDATE_BUDGET_CHANGED")
        for row, prediction in zip(data["validation"], id_ranked):
            prediction["segmentId"] = row["segmentId"]
        id_original = shared["evaluate_rankings"](id_ranked)
        baseline_error = verify_baseline(id_original, reference, id_loss, id_history)
        print("ID baseline reproduced. Starting shared scorer, not another ID model.")
        id_probability = predict_probabilities(id_model, validation_id, len(vocabulary))
        id_masked = masked_id_rankings(data["validation"], id_probability, vocabulary, candidates["validation"])
        train_pack = pack_candidates(data["train"], candidates["train"])
        validation_pack = pack_candidates(data["validation"], candidates["validation"])
        del candidates
        gc.collect()
        candidate_scaler = scale_candidate_inputs(train_pack, validation_pack)
        check_board_piece_scalers(id_scaler, candidate_scaler)
        started = time.perf_counter()
        scorer, scorer_loss, scorer_history = fit_shared_scorer(train_pack, validation_pack, FULL_EPOCHS)
        print(f"Shared fit/epoch-monitoring elapsed: {time.perf_counter() - started:.2f}s (non-canonical timing)")
        shared_ranked = binary_rankings(validation_pack, predict_positive(scorer, validation_pack["x"]))
    predictions = dict(zip(STRATEGIES, (id_ranked, id_masked, shared_ranked)))
    for rows in predictions.values():
        for row, prediction in zip(data["validation"], rows):
            prediction["baselineTargetOov"] = row["targetKey"] not in key_to_id
    results = {name: topk_result(rows) for name, rows in predictions.items()}
    a, b = results["id_candidate_masked"]["gameMacro"], results["shared_candidate_score"]["gameMacro"]
    delta = {k: b[k] - a[k] for k in a}
    status = ("candidate_scoring_signal_observed" if delta["top1"] > 1e-12 and delta["mrr"] >= -1e-12
              else "candidate_scoring_tradeoff_requires_review" if delta["top1"] > 1e-12
              else "no_clear_candidate_scoring_gain")
    oov_mask = [r["targetKey"] not in key_to_id for r in data["validation"]]
    by_oov = {name: [{"baselineTargetOov": flag,
                     **topk_result([r for r, is_oov in zip(rows, oov_mask) if is_oov == flag])}
                    for flag in (False, True)] for name, rows in predictions.items()}
    require(all(sha256_file(p) == digest for p, digest in watched.items())
            and definition_sha(SOURCE_NOTEBOOK) == source_sha
            and definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_SHA, "INPUT_OR_DEFINITIONS_CHANGED")
    report = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "stage": "model_comparison",
              "runMode": "full", "fullCohortCovered": True, "policyDecisionEligible": True,
              "canonicalResultEligible": False, "productionIntegration": False, "inferenceEligible": False,
              "wholePolicyEvidence": False, "historicalRulesetVerified": False,
              "reachableCandidatesGenerated": False, "gameOverSafetyChecked": False,
              "testFileRead": False, "rawFileRead": False, "nodeInvoked": False,
              "modelFitCount": 2, "smokeFitCount": 0, "gameIdLeakage": 0,
              "validationPayloadRead": True, "sourceUnchanged": True,
              "newConditionRepeatDeterminismChecked": False,
              "analysisStatus": status, "nextDecision": "needs_policy_decision",
              "baselineReproduced": True, "baselineMaxMetricError": baseline_error,
              "results": results, "predictions": predictions, "byBaselineOov": by_oov,
              "primaryDelta": delta, "idOriginalMetrics": id_original,
              "epochMetrics": {"id_unmasked": id_history, "shared_candidate_score": scorer_history},
              "optimizationLoss": {"id_unmasked": id_loss, "shared_candidate_score": scorer_loss},
              "cohort": {"trainGames": 149, "validationGames": 31, "trainSegments": 6417,
                         "validationSegments": 1516, "trainCandidateRows": 147450,
                         "validationCandidateRows": 34552, "excludedByCandidateCoverage": 0, "validationOov": 152},
              "training": {"epochs": FULL_EPOCHS, "seed": 1729, "hiddenSizes": [64], "batchSize": 128,
                  "optimizer": "adam", "learningRate": 0.001, "alpha": 0.0001,
                  "inputDimensions": {"id": 207, "shared": 407},
                  "outputClasses": {"id": len(vocabulary), "shared": 2},
                  "sharedBinaryOutputNote": "sklearn binary classifier uses one logistic output unit",
                  "sampleWeightTotal": float(train_pack["weight"].sum()),
                  "scalerWeightTotal": float(train_pack["scalerWeight"].sum()),
                  "decisionWeightMin": float(np.add.reduceat(train_pack["weight"], train_pack["offsets"][:-1]).min()),
                  "decisionWeightMax": float(np.add.reduceat(train_pack["weight"], train_pack["offsets"][:-1]).max()),
                  "sampleWeightMin": float(train_pack["weight"].min()),
                  "sampleWeightMax": float(train_pack["weight"].max()),
                  "globalWeightRescale": False, "weightedRegularization": "alpha_divided_by_minibatch_weight_sum",
                  "validationUsedForFit": False, "validationUsedForModelSelection": False,
                  "convergenceAssessed": False, "fitWarnings": 0, "blasThreads": 1,
                  "rowsPresented": {"id": len(y) * FULL_EPOCHS,
                                    "shared": len(train_pack["y"]) * FULL_EPOCHS},
                  "updates": {"id": math.ceil(len(y) / 128) * FULL_EPOCHS,
                              "shared": math.ceil(len(train_pack["y"]) / 128) * FULL_EPOCHS},
                  "parameterCounts": {"id": sum(x.size for x in (*id_model.coefs_, *id_model.intercepts_)),
                                      "shared": sum(x.size for x in (*scorer.coefs_, *scorer.intercepts_))},
                  "sharedDenseMatrixBytes": int(train_pack["x"].nbytes + validation_pack["x"].nbytes),
                  "idScalerMean": id_scaler.mean_.tolist(), "idScalerScale": id_scaler.scale_.tolist(),
                  "sharedScalerMean": candidate_scaler.mean_.tolist(),
                  "sharedScalerScale": candidate_scaler.scale_.tolist()},
              "provenance": {"coverageSha256": COVERAGE_SHA, "candidateEvidenceSha256": EVIDENCE_SHA,
                  "referenceSha256": REFERENCE_SHA, "snapshotDefinitionSha256": SNAPSHOT_SHA,
                  "sourceDefinitionSha256": source_sha, "manifestSha256": shared["MANIFEST_SHA"],
                  "datasetSha256": {s: shared["EXPECTED"][s][-1] for s in EXPECTED_ROWS},
                  "coreFilesAtCoverage": coverage["provenance"]["coreFiles"]},
              "environment": {"python": platform.python_version(), "numpy": np.__version__,
                              "sklearn": sklearn.__version__, "pandas": pd.__version__},
              "limits": ["same_development_validation_reused", "single_seed_not_statistical_superiority",
                         "static_candidates_not_reachable_actions", "binary_bce_not_multiclass_nll",
                         "not_compute_or_parameter_matched", "no_model_artifact_or_runtime_promotion"]}
    json_bytes(report)
    publish_report(report, output)
    return report


In [ ]:
def check_model_report(report):
    require(report["reportSchemaVersion"] == 1 and report["experimentId"] == EXPERIMENT_ID
            and report["stage"] == "model_comparison" and report["runMode"] == "full"
            and report["fullCohortCovered"] is True and report["policyDecisionEligible"] is True
            and report["modelFitCount"] == 2
            and report["smokeFitCount"] == 0 and report["baselineReproduced"] is True
            and report["gameIdLeakage"] == 0 and report["baselineMaxMetricError"] <= 1e-12,
            "MODEL_REPORT_CONTRACT_INVALID")
    for field in ("canonicalResultEligible", "productionIntegration", "inferenceEligible",
                  "wholePolicyEvidence", "historicalRulesetVerified", "reachableCandidatesGenerated",
                  "gameOverSafetyChecked", "testFileRead", "rawFileRead", "nodeInvoked"):
        require(report[field] is False, "MODEL_REPORT_FLAGS_INVALID")
    p = report["provenance"]
    require(p["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and p["snapshotDefinitionSha256"] == SNAPSHOT_SHA
            and definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_SHA
            and p["coverageSha256"] == COVERAGE_SHA and p["candidateEvidenceSha256"] == EVIDENCE_SHA
            and p["referenceSha256"] == REFERENCE_SHA and p["manifestSha256"] == shared["MANIFEST_SHA"]
            and p["datasetSha256"] == {s: shared["EXPECTED"][s][-1] for s in EXPECTED_ROWS},
            "STALE_MODEL_REPORT")
    require(report["cohort"] == {"trainGames": 149, "validationGames": 31, "trainSegments": 6417,
        "validationSegments": 1516, "trainCandidateRows": 147450, "validationCandidateRows": 34552,
        "excludedByCandidateCoverage": 0, "validationOov": 152}, "MODEL_REPORT_COHORT_INVALID")
    expected_ids = [r["segmentId"] for r in report["predictions"]["id_unmasked"]]
    require(len(set(expected_ids)) == 1516, "REPORT_DUPLICATE_SEGMENTS")
    for name in STRATEGIES:
        rows = report["predictions"][name]
        require([r["segmentId"] for r in rows] == expected_ids
                and len(rows) == 1516 and len({r["gameId"] for r in rows}) == 31
                and all(r["rank"] is None or type(r["rank"]) is int and r["rank"] > 0 for r in rows),
                "MODEL_REPORT_PREDICTIONS_INVALID")
        require(report["results"][name] == topk_result(rows), "MODEL_REPORT_AGGREGATION_INVALID")
        require(sum(r["baselineTargetOov"] for r in rows) == 152 and
                report["byBaselineOov"][name] == [{"baselineTargetOov": flag,
                    **topk_result([r for r in rows if r["baselineTargetOov"] == flag])}
                    for flag in (False, True)], "MODEL_REPORT_OOV_INVALID")
    a, b = report["results"]["id_candidate_masked"]["gameMacro"], report["results"]["shared_candidate_score"]["gameMacro"]
    require(report["primaryDelta"] == {k: b[k] - a[k] for k in a}, "MODEL_REPORT_DELTA_INVALID")
    require(all(len(report["epochMetrics"][name]) == 30 and
                len(report["optimizationLoss"][name]) == 30
                for name in ("id_unmasked", "shared_candidate_score")), "MODEL_REPORT_EPOCHS_INVALID")
    t = report["training"]
    require(t["epochs"] == 30 and t["fitWarnings"] == 0 and t["validationUsedForFit"] is False
            and t["validationUsedForModelSelection"] is False and t["globalWeightRescale"] is False
            and np.isclose(t["sampleWeightTotal"], 6417)
            and np.isclose(t["decisionWeightMin"], 1) and np.isclose(t["decisionWeightMax"], 1),
            "MODEL_REPORT_TRAINING_INVALID")
    expected_status = ("candidate_scoring_signal_observed" if report["primaryDelta"]["top1"] > 1e-12
        and report["primaryDelta"]["mrr"] >= -1e-12 else "candidate_scoring_tradeoff_requires_review"
        if report["primaryDelta"]["top1"] > 1e-12 else "no_clear_candidate_scoring_gain")
    require(report["analysisStatus"] == expected_status, "MODEL_REPORT_DECISION_INVALID")
    return report

def show_model_report(report):
    check_model_report(report)
    print(report["analysisStatus"], "| development comparison, NOT gameplay or whole-policy evidence")
    display(pd.DataFrame([{"strategy": k, **v["gameMacro"], **{"segments": v["segmentMicro"]["segments"]}}
                          for k, v in report["results"].items()]).set_index("strategy"))
    display(pd.DataFrame([{"strategy": name, "baselineTargetOov": row["baselineTargetOov"],
                          **row["gameMacro"], "segments": row["segmentMicro"]["segments"]}
                          for name, groups in report["byBaselineOov"].items() for row in groups]))
    per_game = pd.DataFrame([{"strategy": name, **row} for name, result in report["results"].items()
                             for row in result["perGame"]])
    display(per_game.pivot(index="gameId", columns="strategy", values="top1"))
    t = report["training"]
    display(pd.DataFrame([{"model": name, "parameters": t["parameterCounts"][name],
                          "presentedRows": t["rowsPresented"][name], "updates": t["updates"][name]}
                          for name in ("id", "shared")]))
    id_history = report["epochMetrics"]["id_unmasked"]
    scorer_history = report["epochMetrics"]["shared_candidate_score"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for split in ("train", "validation"):
        axes[0].plot([r["epoch"] for r in id_history],
                     [r[split]["knownTargetNll"] for r in id_history], label=split)
        axes[1].plot([r["epoch"] for r in scorer_history],
                     [r[split]["weightedBinaryNll"] for r in scorer_history], label=split)
        axes[2].plot([r["epoch"] for r in scorer_history],
                     [r[split]["top1"] for r in scorer_history], label=split)
    axes[0].set(title="ID known-target NLL (not BCE)", xlabel="Epoch", ylabel="NLL")
    axes[1].set(title="Shared decision-balanced BCE", xlabel="Epoch", ylabel="Weighted BCE")
    axes[2].set(title="Shared placement Top-1, segment micro", xlabel="Epoch", ylabel="Top-1", ylim=(0, 1))
    for ax in axes:
        ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 3. 내부 검사 — 실제 학습 대신 경계만 검사

아래는 손계산·가상 estimator로 weight 전달/학습 반복·ranking을 확인합니다. 실제 MLP/Scaler fit이나 실제 dataset 읽기는 기본 전체 실행에 포함하지 않습니다. Full 실행에서만 실제 모델을 학습합니다.


In [ ]:
def internal_checks():
    from unittest.mock import patch
    def candidate(col):
        cells = [{"row": r, "col": c} for r in (18, 19) for c in (col, col + 1)]
        pose = {"rotation": 0, "row": 18, "col": col}
        return {"key": physical_key({"pieceType": "square", **pose, "cells": cells}, selected=True),
                "representative": pose, "cells": cells, "aliasCount": 4}
    candidates = [candidate(i) for i in range(3)]
    x = np.array([0.] * 200 + [float(p == "square") for p in shared["PIECES"]])
    row = {"segmentId": "train", "gameId": "train", "pieceType": "square",
           "split": "train", "x": x, "targetKey": candidates[1]["key"]}
    train = pack_candidates([row], [candidates])
    val_row = {**row, "segmentId": "val", "gameId": "val", "split": "validation"}
    validation = pack_candidates([val_row], [candidates])
    assert np.allclose(train["weight"], [.25, .5, .25]) and np.allclose(train["scalerWeight"], 1 / 3)
    assert train["x"].shape == (3, 407) and np.array_equal(train["y"], [0, 1, 0])
    changed = pack_candidates([{**row, "targetKey": candidates[2]["key"]}], [candidates])
    assert np.array_equal(changed["x"], train["x"])
    assert np.array_equal(changed["scalerWeight"], train["scalerWeight"])
    assert np.isclose(binary_nll(train, np.full(3, .5)), np.log(2))
    ranked = binary_rankings(train, np.array([.1, .8, .2]))
    assert ranked[0]["rank"] == 1 and topk_result(ranked)["gameMacro"]["top1"] == 1
    tie = scored_record(row, [candidates[1]["key"], candidates[0]["key"]], np.array([.5, .5]))
    assert tie["predictedKey"] == candidates[0]["key"]
    masked = masked_id_rankings([val_row], np.array([[.9, .1]]),
                                [candidates[0]["key"], candidates[2]["key"]], [candidates])
    assert masked[0]["rank"] is None and topk_result(masked)["segmentMicro"]["top1"] == 0
    try:
        binary_rankings(train, np.array([np.nan, .5, .5]))
        raise AssertionError("NaN accepted")
    except ValueError:
        pass
    class FakeMLP:
        def __init__(self, **kwargs):
            self.calls, self.t_, self.classes_ = [], 0, np.array([0, 1])
            self.coefs_, self.intercepts_ = [np.zeros((407, 1))], [np.zeros(1)]
        def partial_fit(self, a, y, sample_weight=None, classes=None):
            assert np.array_equal(a, train["x"]) and np.array_equal(sample_weight, train["weight"])
            assert classes is None or np.array_equal(classes, [0, 1])
            self.calls.append(len(a)); self.t_ += len(a); self.loss_ = np.float64(.5)
            return self
        def predict_proba(self, a):
            return np.tile([.5, .5], (len(a), 1))
    class FakeScaler:
        def fit(self, a, sample_weight=None):
            assert a is train["x"] and np.array_equal(sample_weight, train["scalerWeight"])
            self.mean_, self.scale_ = np.zeros(407), np.ones(407)
            return self
        def transform(self, a, copy=False):
            assert a is train["x"] or a is validation["x"]
            return a
    with patch.dict(globals(), {"MLPClassifier": FakeMLP, "StandardScaler": FakeScaler}), patch("builtins.print"):
        scaler = scale_candidate_inputs(train, validation)
        model, loss, history = fit_shared_scorer(train, validation, 2)
        assert model.calls == [3, 3] and len(history) == len(loss) == 2
    baseline_scaler = copy.deepcopy(scaler)
    baseline_scaler.mean_ = np.zeros(207)
    baseline_scaler.scale_ = np.ones(207)
    scaler.mean_[168] = 1.34e-12
    scaler.scale_[143] += 9.5e-13
    check_board_piece_scalers(baseline_scaler, scaler)
    for field in ("mean_", "scale_"):
        changed_scaler = copy.deepcopy(scaler)
        getattr(changed_scaler, field)[168] += 1e-6
        try:
            check_board_piece_scalers(baseline_scaler, changed_scaler)
            raise AssertionError("real scaler drift accepted")
        except ValueError as error:
            assert "BOARD_PIECE_SCALER_WEIGHTING_CHANGED" in str(error)
    changed_scaler = copy.deepcopy(scaler)
    changed_scaler.mean_[0] = np.nan
    try:
        check_board_piece_scalers(baseline_scaler, changed_scaler)
        raise AssertionError("nonfinite scaler accepted")
    except ValueError:
        pass
    with tempfile.TemporaryDirectory(dir="/private/tmp", prefix="candidate-model-check-") as directory:
        root = Path(directory); existing = root / "keep.json"; existing.write_text("keep")
        link = root / "link.json"; link.symlink_to(existing)
        hard = root / "hard.json"; os.link(existing, hard)
        dangling = root / "dangling.json"; dangling.symlink_to(root / "missing.json")
        for path in (existing, link, hard, dangling, root / ".." / root.name / "alias.json", SOURCE_NOTEBOOK):
            try:
                report_path(str(path))
                raise AssertionError("unsafe output accepted")
            except ValueError:
                pass
    # Validate the report reader with a local unit fixture; never publish/display synthetic performance.
    predictions = {k: [] for k in STRATEGIES}
    for i in range(1516):
        example = {**row, "segmentId": f"fixture{i}", "gameId": f"g{i % 31}"}
        for name in STRATEGIES:
            prediction = scored_record(example, [c["key"] for c in candidates], np.array([.1, .8, .2]))
            prediction["baselineTargetOov"] = i >= 1364
            if i >= 1364 and name != "shared_candidate_score":
                prediction["rank"], prediction["knownTarget"] = None, False
            predictions[name].append(prediction)
    results = {k: topk_result(v) for k, v in predictions.items()}
    a, b = results["id_candidate_masked"]["gameMacro"], results["shared_candidate_score"]["gameMacro"]
    fixture = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "stage": "model_comparison",
        "runMode": "full", "fullCohortCovered": True, "policyDecisionEligible": True,
        "modelFitCount": 2, "smokeFitCount": 0, "baselineReproduced": True,
        "gameIdLeakage": 0, "baselineMaxMetricError": 0.,
        **{k: False for k in ("canonicalResultEligible", "productionIntegration", "inferenceEligible",
            "wholePolicyEvidence", "historicalRulesetVerified", "reachableCandidatesGenerated",
            "gameOverSafetyChecked", "testFileRead", "rawFileRead", "nodeInvoked")},
        "provenance": {"sourceDefinitionSha256": definition_sha(SOURCE_NOTEBOOK),
            "snapshotDefinitionSha256": SNAPSHOT_SHA, "coverageSha256": COVERAGE_SHA,
            "candidateEvidenceSha256": EVIDENCE_SHA, "referenceSha256": REFERENCE_SHA,
            "manifestSha256": shared["MANIFEST_SHA"],
            "datasetSha256": {s: shared["EXPECTED"][s][-1] for s in EXPECTED_ROWS}},
        "cohort": {"trainGames": 149, "validationGames": 31, "trainSegments": 6417,
            "validationSegments": 1516, "trainCandidateRows": 147450, "validationCandidateRows": 34552,
            "excludedByCandidateCoverage": 0, "validationOov": 152},
        "predictions": predictions, "results": results,
        "byBaselineOov": {k: [{"baselineTargetOov": flag,
            **topk_result([r for r in v if r["baselineTargetOov"] == flag])}
            for flag in (False, True)] for k, v in predictions.items()},
        "primaryDelta": {k: b[k] - a[k] for k in a},
        "analysisStatus": "candidate_scoring_signal_observed",
        "epochMetrics": {name: [{"epoch": i+1, "train": {"knownTargetNll": .7,
            "weightedBinaryNll": .7, "top1": .5}, "validation": {"knownTargetNll": .8,
            "weightedBinaryNll": .8, "top1": .4}} for i in range(30)]
            for name in ("id_unmasked", "shared_candidate_score")},
        "optimizationLoss": {name: [.7]*30 for name in ("id_unmasked", "shared_candidate_score")},
        "training": {"epochs": 30, "fitWarnings": 0, "validationUsedForFit": False,
            "validationUsedForModelSelection": False, "globalWeightRescale": False,
            "sampleWeightTotal": 6417., "decisionWeightMin": 1., "decisionWeightMax": 1.,
            "parameterCounts": {"id": 151957, "shared": 26177},
            "rowsPresented": {"id": 192510, "shared": 4423500},
            "updates": {"id": 1530, "shared": 34560}}}
    check_model_report(fixture)
    for field in ("modelFitCount", "productionIntegration", "analysisStatus"):
        changed_report = copy.deepcopy(fixture)
        changed_report[field] = 0 if field == "modelFitCount" else True if field == "productionIntegration" else "bad"
        try:
            check_model_report(changed_report)
            raise AssertionError("tampered report accepted")
        except ValueError:
            pass
    scores = {"top1": .5, "top3": .8, "mrr": .65, "knownTargetNll": .9}
    history = [{"epoch": i+1, "train": dict(scores), "validation": dict(scores)} for i in range(30)]
    baseline = {"gameMacro": {k: scores[k] for k in ("top1", "top3", "mrr")}, "segmentMicro": dict(scores)}
    reference = {"results": {"board_piece": copy.deepcopy(baseline)},
                 "trainingLoss": {"board_piece": [.9]*30}, "epochMetrics": {"board_piece": history}}
    assert verify_baseline(baseline, reference, [.9]*30, history) == 0
    baseline["gameMacro"]["top1"] += .1
    try:
        verify_baseline(baseline, reference, [.9]*30, history)
        raise AssertionError("baseline drift accepted")
    except ValueError:
        pass
    print("PASS: input/label separation, decision weights, BCE, OOV/tie, fake optimizer/scaler, output protection; real fit=0")

internal_checks()


## 4. Full — 실제 두 모델 학습

`tetris-ml` 커널에서 실행합니다. Node 설정은 필요 없습니다. `RUN_FULL=True`와 아직 없는 `FULL_REPORT_PATH`를 지정하고 설정→실행→결과 순서로 실행하세요.

먼저 ID 모델30epoch를 학습해 과거 기준선을 재현합니다. 실패하면 scorer 학습 전에 중단합니다. 통과하면 공유 scorer30epoch를 학습합니다. Validation loss와 placement 지표는 매 epoch 관찰하지만 early stopping/epoch 선택에 쓰지 않습니다.

공유 모델은 Train147,450/Validation34,552개 후보 행을 읽습니다. 두407차원 float64 행렬만 약565MiB이며 evidence 파싱·scaler·학습 중 메모리 사용은 더 큽니다.30회 전체 pass는 ID 모델과 계산량이 같지 않고 수분 이상 걸릴 수 있습니다. 계산 시간 수치는 사전 보장하지 않습니다.

선택 후보별 이진 BCE와 ID 다중분류 NLL은 다른 손실입니다. 같은 alpha라도 sample-weight 합에 따른 sklearn L2 정규화가 다르므로 순수한 출력층 변경만의 인과 실험으로 읽지 마세요.


In [ ]:
RUN_FULL = False
FULL_REPORT_PATH = os.environ.get("HUMAN_CANDIDATE_MODEL_REPORT")


In [ ]:
FULL_REPORT = None
if RUN_FULL:
    FULL_REPORT = run_model_comparison(FULL_REPORT_PATH)
    print(FULL_REPORT["analysisStatus"])
else:
    print("Full skipped. No real input load or MLP/scaler fit. Both training implementations are available above.")


In [ ]:
if FULL_REPORT is not None:
    show_model_report(FULL_REPORT)
else:
    print("No Full result yet. Internal fake-estimator checks are not model performance.")


## 5. 저장한 결과 읽기

재학습 없이 report만 읽습니다. `REPORT_TO_READ`에 생성한 경로를 지정합니다. 과거 coverage Notebook을 다시 실행할 필요는 없습니다.


In [ ]:
REPORT_TO_READ = None
SAVED_REPORT = None
if REPORT_TO_READ is not None:
    path = repo_input(REPORT_TO_READ)
    SAVED_REPORT = json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_constant,
                              parse_float=finite_float)
    show_model_report(SAVED_REPORT)
else:
    print("Set REPORT_TO_READ to display the saved model comparison.")


## 6. 해석과 다음 gate

Primary는 Validation31게임의 game-macro Top-1에서 shared scorer−후보 제한 ID의 차이입니다. 원래 ID와 후보 제한 ID의 차이는 같은 모델에서 후보 제약만의 효과입니다. OOV152개도 전체 분모에 남깁니다.

Train/Validation BCE가 함께 낮아지는지, Train만 좋아지는지와 placement Top-1/MRR을 함께 보세요. 이진 후보 정확도는 음성 후보가 많아 높게 나올 수 있어 primary로 쓰지 않습니다. Epoch는30으로 고정하며 결과를 보고 최적 epoch를 자동 채택하지 않습니다.

기존 Validation 반복 사용·단일 seed·후보 geometry/reachability 미검증·모델/행 구성 차이가 남습니다. 공유 scorer가 좋아져도 전체 Human 조작·리듬·gameplay 정책을 완성한 것은 아닙니다. Full 결과 해석 후 다음 판단을 요청하며 runtime·새 수집·재학습·commit을 자동 진행하지 않습니다.
